In [1]:
import os
for root, dirs, files in os.walk("/kaggle/input"):
    level = root.replace("/kaggle/input", "").count(os.sep)
    if level <= 3:
        print(root)

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/zeinebbayoudh7
/kaggle/input/datasets/zeinebbayoudh7/emotions-final


In [3]:
import os, random, warnings
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import classification_report, accuracy_score, recall_score
from PIL import Image
from tqdm import tqdm

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")

BASE_DIR   = Path("/kaggle/input/datasets/zeinebbayoudh7/emotions-final")
OUTPUT_DIR = Path("/kaggle/working/emotions_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LABELS_CSV = BASE_DIR / "labels.csv"

CLASSES      = ["happy","relaxed","neutral","sad","frown","alert","angry"]
NUM_CLASSES  = len(CLASSES)
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
GROUPE_MAP   = {"happy":"normal","relaxed":"normal","neutral":"normal",
                "sad":"normal","frown":"normal","alert":"anormal","angry":"anormal"}

IMG_SIZE=224; BATCH_SIZE=32
P1_LR=1e-3;  P1_EPOCHS=15
P2_LR=1e-5;  P2_EPOCHS=30
DROPOUT=0.6; LABEL_SMOOTHING=0.1; WEIGHT_DECAY=1e-3
ES_PATIENCE_P1=5; ES_PATIENCE_P2=7

print(f"Classes : {CLASSES}")
print(f"Output  : {OUTPUT_DIR}")
print("✅ Cellule 1 OK")

Device : cuda
Classes : ['happy', 'relaxed', 'neutral', 'sad', 'frown', 'alert', 'angry']
Output  : /kaggle/working/emotions_output
✅ Cellule 1 OK


In [4]:
# CELLULE 2 — Chargement labels.csv
df = pd.read_csv(LABELS_CSV)
df["label_idx"] = df["label"].map(CLASS_TO_IDX)

print(f"Total : {len(df)} images")
print(f"\nSplit :\n{df['split'].value_counts()}")
print(f"\nLabels :\n{df['label'].value_counts()}")
print(f"\nNaN : {df.isnull().sum().sum()}")
print("\n✅ Cellule 2 OK")

Total : 28280 images

Split :
split
train    24116
test      2083
val       2081
Name: count, dtype: int64

Labels :
label
happy      4040
relaxed    4040
neutral    4040
sad        4040
frown      4040
alert      4040
angry      4040
Name: count, dtype: int64

NaN : 0

✅ Cellule 2 OK


In [5]:
# CELLULE 3 — Dataset PyTorch
class EmotionDataset(Dataset):
    def __init__(self, df, base_dir, transform):
        self.df        = df.reset_index(drop=True)
        self.base_dir  = Path(base_dir)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        img   = Image.open(self.base_dir / row["filepath"]).convert("RGB")
        label = row["label_idx"]
        return self.transform(img), label

# Transformations
train_tf = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(0.3, 0.3, 0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

val_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

# DataLoaders
train_df = df[df["split"]=="train"]
val_df   = df[df["split"]=="val"]
test_df  = df[df["split"]=="test"]

train_loader = DataLoader(EmotionDataset(train_df, BASE_DIR, train_tf),
                          batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(EmotionDataset(val_df,   BASE_DIR, val_tf),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(EmotionDataset(test_df,  BASE_DIR, val_tf),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train : {len(train_df)} | Val : {len(val_df)} | Test : {len(test_df)}")
print(f"Batches train : {len(train_loader)}")
print("✅ Cellule 3 OK")

Train : 24116 | Val : 2081 | Test : 2083
Batches train : 754
✅ Cellule 3 OK


In [6]:
def train_model(model, train_loader, val_loader, epochs, lr, patience, phase):
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_val_loss = float('inf')
    best_val_acc  = 0.0
    best_weights  = None
    no_improve    = 0
    history       = []

    print(f"🚀 {phase} — lr={lr} | epochs={epochs} | patience={patience}\n")

    for epoch in range(1, epochs+1):
        model.train()
        train_loss, train_correct = 0, 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            out  = model(imgs)
            loss = criterion(out, labels)
            loss.backward()
            optimizer.step()
            train_loss    += loss.item() * imgs.size(0)
            train_correct += (out.argmax(1) == labels).sum().item()

        model.eval()
        val_loss, val_correct = 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                out  = model(imgs)
                loss = criterion(out, labels)
                val_loss    += loss.item() * imgs.size(0)
                val_correct += (out.argmax(1) == labels).sum().item()

        train_loss /= len(train_loader.dataset)
        val_loss   /= len(val_loader.dataset)
        train_acc   = train_correct / len(train_loader.dataset)
        val_acc     = val_correct   / len(val_loader.dataset)

        scheduler.step()

        history.append({"epoch":epoch, "train_loss":train_loss,
                        "val_loss":val_loss, "train_acc":train_acc, "val_acc":val_acc})

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_val_acc  = val_acc
            best_weights  = model.state_dict().copy()
            no_improve    = 0
            flag = "💾"
        else:
            no_improve += 1
            flag = f"⏳ {no_improve}/{patience}"

        print(f"  Epoch {epoch:2d}/{epochs} | "
              f"Train Loss: {train_loss:.4f} Acc: {train_acc*100:.1f}% | "
              f"Val Loss: {val_loss:.4f} Acc: {val_acc*100:.1f}% {flag}")

        if no_improve >= patience:
            print(f"\n  ⏹ Early stopping epoch {epoch}")
            break

    model.load_state_dict(best_weights)
    print(f"\n✅ {phase} terminée — Meilleure Val Acc: {best_val_acc*100:.1f}%")
    return model, history

print("✅ Cellule 4 OK")

✅ Cellule 4 OK


In [7]:
model_eff = models.efficientnet_b0(weights="IMAGENET1K_V1")
for p in model_eff.parameters():
    p.requires_grad = False
model_eff.classifier = nn.Sequential(
    nn.Dropout(DROPOUT),
    nn.Linear(model_eff.classifier[1].in_features, NUM_CLASSES)
)
model_eff = model_eff.to(DEVICE)


total  = sum(p.numel() for p in model_eff.parameters())
train  = sum(p.numel() for p in model_eff.parameters() if p.requires_grad)
frozen = total - train

print(f"Total params    : {total:,}")
print(f"Entraînables    : {train:,}  ← tête seulement (Phase 1)")
print(f"Gelés           : {frozen:,} ← backbone ImageNet")
print("✅ Cellule 5a OK")

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 132MB/s] 


Total params    : 4,016,515
Entraînables    : 8,967  ← tête seulement (Phase 1)
Gelés           : 4,007,548 ← backbone ImageNet
✅ Cellule 5a OK


In [8]:
# CELLULE 6a — Phase 1 EfficientNetB0 (tête seulement)
print("🚀 EfficientNetB0 — Phase 1\n")

model_eff, history_eff_p1 = train_model(
    model      = model_eff,
    train_loader = train_loader,
    val_loader   = val_loader,
    epochs     = P1_EPOCHS,
    lr         = P1_LR,
    patience   = ES_PATIENCE_P1,
    phase      = "EFF-P1"
)

# Sauvegarder
torch.save(model_eff.state_dict(), OUTPUT_DIR / "efficientnet_p1.pth")
print(f"\n💾 Modèle sauvegardé")
print("✅ Cellule 6a OK")

🚀 EfficientNetB0 — Phase 1

🚀 EFF-P1 — lr=0.001 | epochs=15 | patience=5

  Epoch  1/15 | Train Loss: 1.5445 Acc: 45.5% | Val Loss: 1.6521 Acc: 40.2% 💾
  Epoch  2/15 | Train Loss: 1.4792 Acc: 49.4% | Val Loss: 1.6371 Acc: 39.6% 💾
  Epoch  3/15 | Train Loss: 1.4757 Acc: 49.7% | Val Loss: 1.6343 Acc: 41.3% 💾
  Epoch  4/15 | Train Loss: 1.4733 Acc: 49.9% | Val Loss: 1.6382 Acc: 40.6% ⏳ 1/5
  Epoch  5/15 | Train Loss: 1.4681 Acc: 50.1% | Val Loss: 1.6303 Acc: 41.3% 💾
  Epoch  6/15 | Train Loss: 1.4690 Acc: 49.9% | Val Loss: 1.6287 Acc: 41.4% 💾
  Epoch  7/15 | Train Loss: 1.4574 Acc: 50.4% | Val Loss: 1.6226 Acc: 41.5% 💾
  Epoch  8/15 | Train Loss: 1.4571 Acc: 50.5% | Val Loss: 1.6252 Acc: 41.9% ⏳ 1/5
  Epoch  9/15 | Train Loss: 1.4448 Acc: 51.2% | Val Loss: 1.6189 Acc: 41.4% 💾
  Epoch 10/15 | Train Loss: 1.4374 Acc: 51.4% | Val Loss: 1.6156 Acc: 41.2% 💾
  Epoch 11/15 | Train Loss: 1.4324 Acc: 51.4% | Val Loss: 1.6130 Acc: 41.3% 💾
  Epoch 12/15 | Train Loss: 1.4262 Acc: 51.8% | Val Loss: 1.

In [9]:
torch.save(model_eff.state_dict(), OUTPUT_DIR / "efficientnet_p1.pth")
print("💾 Phase 1 sauvegardée")

💾 Phase 1 sauvegardée


In [1]:
# Réexécute cellules 1,2,3,4,5a puis :
model_eff.load_state_dict(torch.load(OUTPUT_DIR / "efficientnet_p1.pth"))
print("✅ Poids Phase 1 rechargés")

✅ Poids Phase 1 rechargés


In [2]:
# CELLULE 6b — Phase 2 EfficientNetB0 (fine-tuning partiel)

# Recharger les poids Phase 1
model_eff.load_state_dict(torch.load(OUTPUT_DIR / "efficientnet_p1.pth"))

# Dégeler seulement les 3 derniers blocs + tête
for p in model_eff.parameters():
    p.requires_grad = False

for p in model_eff.features[-3:].parameters():
    p.requires_grad = True
for p in model_eff.classifier.parameters():
    p.requires_grad = True

train = sum(p.numel() for p in model_eff.parameters() if p.requires_grad)
print(f"Paramètres entraînables : {train:,}\n")

model_eff, history_eff_p2 = train_model(
    model        = model_eff,
    train_loader = train_loader,
    val_loader   = val_loader,
    epochs       = P2_EPOCHS,
    lr           = P2_LR,
    patience     = ES_PATIENCE_P2,
    phase        = "Phase 2 — EfficientNetB0"
)

torch.save(model_eff.state_dict(), OUTPUT_DIR / "efficientnet_p2.pth")
print("💾 Modèle sauvegardé")

Paramètres entraînables : 3,164,707

🚀 Phase 2 — EfficientNetB0 — lr=1e-05 | epochs=30 | patience=7

  Epoch  1/30 | Train Loss: 1.3745 Acc: 54.4% | Val Loss: 1.5662 Acc: 43.1% 💾
  Epoch  2/30 | Train Loss: 1.3236 Acc: 57.1% | Val Loss: 1.5350 Acc: 44.9% 💾
  Epoch  3/30 | Train Loss: 1.2847 Acc: 58.9% | Val Loss: 1.5165 Acc: 45.7% 💾
  Epoch  4/30 | Train Loss: 1.2617 Acc: 60.4% | Val Loss: 1.5013 Acc: 46.9% 💾
  Epoch  5/30 | Train Loss: 1.2470 Acc: 60.7% | Val Loss: 1.4890 Acc: 46.4% 💾
  Epoch  6/30 | Train Loss: 1.2261 Acc: 61.9% | Val Loss: 1.4762 Acc: 47.0% 💾
  Epoch  7/30 | Train Loss: 1.2092 Acc: 62.8% | Val Loss: 1.4689 Acc: 47.8% 💾
  Epoch  8/30 | Train Loss: 1.2042 Acc: 62.7% | Val Loss: 1.4686 Acc: 48.1% 💾
  Epoch  9/30 | Train Loss: 1.1868 Acc: 63.6% | Val Loss: 1.4587 Acc: 48.2% 💾
  Epoch 10/30 | Train Loss: 1.1816 Acc: 64.0% | Val Loss: 1.4528 Acc: 48.6% 💾
  Epoch 11/30 | Train Loss: 1.1729 Acc: 64.3% | Val Loss: 1.4500 Acc: 48.5% 💾
  Epoch 12/30 | Train Loss: 1.1666 Acc: 6

In [3]:
!pip install ultralytics -q
from ultralytics import YOLO

model_yolo = YOLO("yolov8m-cls.pt")
print("✅ Cellule 5c OK")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 24.6 MB/s eta 0:00:0000:01
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Cellule 5c OK


In [4]:
# CELLULE 6c — Entraînement YOLOv8m-cls

# Préparer le dossier data
DATA_DIR = "/kaggle/input/datasets/zeinebbayoudh7/emotions-final"

# Phase 1
model_yolo.train(
    data    = DATA_DIR,
    epochs  = P1_EPOCHS,
    imgsz   = IMG_SIZE,
    batch   = BATCH_SIZE,
    lr0     = P1_LR,
    dropout = DROPOUT,
    patience= ES_PATIENCE_P1,
    project = str(OUTPUT_DIR),
    name    = "yolo_p1",
    exist_ok= True,
    verbose = True
)

print("✅ YOLOv8 Phase 1 terminée")

Ultralytics 8.4.45 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/datasets/zeinebbayoudh7/emotions-final, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.6, dynamic=False, embed=None, end2end=None, epochs=15, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo_p1, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, ove

In [ ]:
# CELLULE 6d — YOLOv8 Phase 2 (fine-tuning complet)

model_yolo = YOLO(str(OUTPUT_DIR / "yolo_p1/weights/best.pt"))

model_yolo.train(
    data     = DATA_DIR,
    epochs   = P2_EPOCHS,
    imgsz    = IMG_SIZE,
    batch    = BATCH_SIZE,
    lr0      = P2_LR,
    dropout  = DROPOUT,
    patience = ES_PATIENCE_P2,
    project  = str(OUTPUT_DIR),
    name     = "yolo_p2",
    exist_ok = True,
    verbose  = True
)

print("✅ YOLOv8 Phase 2 terminée")

Ultralytics 8.4.45 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/datasets/zeinebbayoudh7/emotions-final, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.6, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=1e-05, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/kaggle/working/emotions_output/yolo_p1/weights/best.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo_p2, nbs=64, nms=False, opset=